## Description

This is mostly derived from `fourcastnext`. It is used to illustrate how to create a similar "inference" pipeline using PET persistence model.

In [1]:
# Most users should change this to the current directory.
# os.environ['ERA5LOWRESDEMO'] = os.path.abspath('/tmp/')

# NOTE: /var/tmp/ is used for longer running tasks and persistence that's required across reboots
import os
os.environ['ERA5LOWRESDEMO'] = os.path.abspath('/var/tmp/era5demo_persistence')
os.makedirs(os.environ['ERA5LOWRESDEMO'], exist_ok=True)
EXPERIMENT_VERSION='v1'



In [2]:

import pathlib
import xarray as xr
from pathlib import Path

# ---
# unsure what these are for:
# ---
# import hydra
# from omegaconf import OmegaConf
# ---

# ---
# we are downloading from the internet. Does this register the archive?
# ---
# import pyearthtools.data
# import pyearthtools.data.archive
# ---

# ---
# training  not required
# ---
# import pyearthtools.training
# import fourcastnext
# ---

import pyearthtools.tutorial
import pyearthtools.pipeline

# ---
# no gpu required
# ---
# import torch; torch.set_default_device(<YOUR_DEVICE_HERE>)  # Uncomment and set this if you need to configure a non-default device.
# ---

In [3]:
workdir = os.environ['ERA5LOWRESDEMO']
print(f'This tutorial will download a copy of the input data to {workdir}. It will also create model checkpoint files and other data here.')

# ---
# this does not work, but it may not be needed
# ---
# if pyearthtools.data.archive.ROOT_DIRECTORIES['era5lowresdemo'] != workdir:
#     print("There is some misconfiguration of your working directory, please review the commented out cells at the start of the notebook")
# ---

file_location = workdir + '/mini.nc'



This tutorial will download a copy of the input data to /var/tmp/era5demo_persistence. It will also create model checkpoint files and other data here.


In [4]:
if not os.path.exists(file_location):
    print("Training data not found, downloading around 2.8GB of data")
    era5_lowres = xr.open_zarr('gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr')
    subset = era5_lowres[['10m_u_component_of_wind', 
                          '10m_v_component_of_wind', 
                          '2m_temperature', 
                          'mean_sea_level_pressure',
                          #'geopotential',  # Uncomment this to fetch additional data
                          #'toa_incident_solar_radiation_6hr', # Uncomment this to fetch additional data
                          #'temperature' # Uncomment this to fetch additional data
                         ]]

    # bilevel = subset.sel({'level': [50, 500]}) Uncomment if fetching addtional data    
    # bilevel.to_netcdf(file_location)

    subset.to_netcdf(file_location)  # Comment this out if using the bilevel data instead
    print(f"Wrote file to {file_location}")
    assert os.path.exists(file_location)
else:
    print(f"File already downloaded ({file_location}), skipping ...")

File already downloaded (/var/tmp/era5demo_persistence/mini.nc), skipping ...


In [5]:
accessor = pyearthtools.tutorial.ERA5DataClass.ERA5LowResDemoIndex([
                 '10m_u_component_of_wind', 
                 '10m_v_component_of_wind', 
                 'mean_sea_level_pressure',
                 '2m_temperature'    
],
filename_override=file_location)

In [28]:
# --- scratch space/workings ---
# 1. data is in 6 hour intervals
# 2. we want the past 6 indices for median to work, so taking 2 days is sufficient without breaking the intervals
# 3. explicitly set the time update to 6 hours since the median will be performed on every index using past 3 indices (but padded to 8 for safety/imputation)
# ---
# SequentialRetrieval works like this:
# (a, b, c)
# a = start
# b = number of values to get
# c = interval or how many values to skip
# ---
# TemporalRetrieval does the same, but each index is a delta-unit mapping - so we need to reverse engineer a bit
# !!! IMPORTANT. The circumstance here is that time is already 6 hourly windows, 
# What this means is this:
# - selecting 1 index =>
# ---
import datetime
import functools
data_pipeline = pyearthtools.pipeline.Pipeline(
    accessor,
    pyearthtools.data.transforms.coordinates.StandardLongitude(type="-180-180"),
    pyearthtools.pipeline.modifications.TemporalWindow(
        prior_indexes=list(range(-6,0,1)),
        posterior_indexes=[0],
        timedelta=datetime.timedelta(hours=6),
        merge_method=functools.partial(xr.concat, dim="time"),
    ),
    iterator=pyearthtools.pipeline.iterators.DateRange(1980, 2016, interval='6h')
)

In [30]:
# works
data_pipeline["2000-05-05"][0]
# does not work
data_pipeline["2000-05-05T00"][0]

DataNotFoundError: Data with args: (Petdt('2000-05-03T12'),) could not be found.